# Crypto Lab 06B — RSA

**CYBR 3570: Applied Cryptography**  
**Topic:** RSA, trapdoor permutations, OAEP, PSS, and implementation risks

## Today's big question

> Why is “uses RSA” not enough information to know whether a system is secure?

RSA is one of the most famous public-key cryptosystems, but the core mathematical operation is only the beginning. This notebook separates toy RSA math from secure library-based RSA usage.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the difference between symmetric-key and public-key encryption.
2. Generate a toy RSA key pair from small primes.
3. Compute Euler's totient `φ(n)` and the private exponent `d`.
4. Demonstrate textbook RSA encryption and decryption on small integers.
5. Explain why textbook RSA is deterministic and malleable.
6. Use RSA-OAEP with a professional cryptographic library.
7. Use RSA-PSS to sign and verify messages.
8. Explain why RSA should not be implemented from scratch in production systems.

## Setup

This notebook uses only standard Python for the toy examples. The production-style examples use the `cryptography` package.

In [1]:
import math
import time
from dataclasses import dataclass
from typing import Tuple, Optional

print("Lab 06B setup complete")

Lab 06B setup complete


## Part 1 — RSA as a trapdoor permutation

RSA operates on integers modulo `n`, where `n = p * q` for two large secret primes.

The public operation is:

$$y = x^e \mod n$$

The private trapdoor operation is:

$$x = y^d \mod n$$

The values `n` and `e` are public. The value `d` is private. The private exponent works because:

$$ed \equiv 1 \pmod{
arphi(n)}$$

where, for RSA moduli `n = pq`,

$$
arphi(n) = (p-1)(q-1)$$

### Exercise 1 — Concept check

In 2–4 sentences, explain why RSA is called a public-key system. Your answer should distinguish the public key from the private key.

Answer:
- A RSA key has a public and a private key. THe public key is shared to encrypt and known to everyone and the private is only for the holder. 

## Part 2 — Building a toy RSA key

The following functions implement the basic math needed for toy RSA.

These examples are intentionally small and insecure. They exist only to make the algebra visible.

In [2]:
def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """Extended Euclidean algorithm.

    Returns (g, x, y) such that ax + by = g = gcd(a, b).
    """
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    return (g, y1, x1 - (a // b) * y1)


def mod_inverse(a: int, m: int) -> int:
    """Return the modular inverse of a modulo m."""
    g, x, _ = egcd(a, m)
    if g != 1:
        raise ValueError(f"{a} has no inverse modulo {m}")
    return x % m


def phi_for_two_primes(p: int, q: int) -> int:
    """Euler's totient for n = p*q where p and q are prime."""
    return (p - 1) * (q - 1)

print("Inverse of 17 mod 3120:", mod_inverse(17, 3120))

Inverse of 17 mod 3120: 2753


### Exercise 2 — Compute toy RSA parameters

Use `p = 61`, `q = 53`, and `e = 17`.

Compute:

- `n = p * q`
- `phi = (p - 1) * (q - 1)`
- `d = e^-1 mod phi`

Then verify that `(e * d) % phi == 1`.

In [4]:
# TODO: Complete this toy RSA setup.
p = 61
q = 53
e = 17

n = p * q
phi = (p - 1) * (q - 1)
d = mod_inverse(e, phi)  

print("n:", n)
print("phi:", phi)
print("d:", d)
print("check:", (d * e) % phi == 1)

n: 3233
phi: 3120
d: 2753
check: True


## Part 3 — Textbook RSA encryption and decryption

Textbook RSA directly applies the trapdoor permutation to a message integer.

Encryption:

$$c = m^e \mod n$$

Decryption:

$$m = c^d \mod n$$

Again: this is educational only. Textbook RSA is not secure encryption.

In [5]:
def rsa_encrypt_textbook(m: int, e: int, n: int) -> int:
    """Toy textbook RSA encryption."""
    if not 0 <= m < n:
        raise ValueError("message must be in range 0 <= m < n")
    return pow(m, e, n)


def rsa_decrypt_textbook(c: int, d: int, n: int) -> int:
    """Toy textbook RSA decryption."""
    return pow(c, d, n)

# Worked example using the parameters from the previous exercise.
p_ex, q_ex, e_ex = 61, 53, 17
n_ex = p_ex * q_ex
phi_ex = phi_for_two_primes(p_ex, q_ex)
d_ex = mod_inverse(e_ex, phi_ex)

m = 65
c = rsa_encrypt_textbook(m, e_ex, n_ex)
recovered = rsa_decrypt_textbook(c, d_ex, n_ex)

print("message:", m)
print("ciphertext:", c)
print("recovered:", recovered)

message: 65
ciphertext: 2790
recovered: 65


### Exercise 3 — Encrypt and decrypt three messages

Using your toy key from Exercise 2, encrypt and decrypt three integer messages smaller than `n`.

Show that decryption recovers the original message each time.

In [7]:
# TODO: Add three integer messages smaller than n.
messages = [100, 420, 100]

for m in messages:
    c = rsa_encrypt_textbook(m, e, n)
    recovered = rsa_decrypt_textbook(c, d, n)
    print(f"m={m}, c={c}, recovered={recovered}, ok={m == recovered}")

m=100, c=1773, recovered=100, ok=True
m=420, c=2694, recovered=420, ok=True
m=100, c=1773, recovered=100, ok=True


## Part 4 — Why textbook RSA is deterministic

A secure encryption scheme should not reveal when the same plaintext has been encrypted twice.

Textbook RSA fails this test because it is deterministic: the same message encrypted under the same public key always produces the same ciphertext.

In [8]:
# Worked example: deterministic encryption
m = 42
c1 = rsa_encrypt_textbook(m, e_ex, n_ex)
c2 = rsa_encrypt_textbook(m, e_ex, n_ex)
print("first encryption: ", c1)
print("second encryption:", c2)
print("same ciphertext?", c1 == c2)

first encryption:  2557
second encryption: 2557
same ciphertext? True


### Exercise 4 — Explain determinism

Why is deterministic public-key encryption a problem when the set of possible messages is small? Give an example scenario.

Answer:
- Because the encrytions will be the same so you can reveal the message through the shared key. Same information should be encrypted differently so we can't see patterns. 

## Part 5 — Textbook RSA is malleable

Textbook RSA has a multiplicative property:

$$Enc(m_1) \cdot Enc(m_2) \equiv Enc(m_1 \cdot m_2) \pmod n$$

That means an attacker can combine ciphertexts to create a new ciphertext related to the original plaintexts, even without knowing the private key.

In [9]:
# Worked example: textbook RSA malleability
m1 = 7
m2 = 9

c1 = rsa_encrypt_textbook(m1, e_ex, n_ex)
c2 = rsa_encrypt_textbook(m2, e_ex, n_ex)

combined_ciphertext = (c1 * c2) % n_ex
combined_plaintext = rsa_decrypt_textbook(combined_ciphertext, d_ex, n_ex)

print("m1:", m1)
print("m2:", m2)
print("m1*m2 mod n:", (m1 * m2) % n_ex)
print("decrypted combined ciphertext:", combined_plaintext)

m1: 7
m2: 9
m1*m2 mod n: 63
decrypted combined ciphertext: 63


### Exercise 5 — Demonstrate malleability yourself

Choose two new messages `m1` and `m2`. Encrypt them separately, multiply the ciphertexts modulo `n`, and decrypt the result.

Confirm that the decrypted value equals `(m1 * m2) % n`.

In [12]:
# TODO: Pick two messages and demonstrate textbook RSA malleability.
m1 = 99
m2 = 999

# Your code here
c1 = rsa_encrypt_textbook(m1, e_ex, n_ex)
c2 = rsa_encrypt_textbook(m2, e_ex, n_ex)

combined_text = (c1 * c2) % n_ex
combined_plaintext = rsa_decrypt_textbook(combined_text, d, n)

print(f"c1={c1}, c2={c2}, combined_text={combined_text}, combined_plaintext={combined_plaintext}")

c1=281, c2=2464, combined_text=522, combined_plaintext=1911


## Part 6 — Fast modular exponentiation

RSA depends on modular exponentiation. A naive implementation would multiply repeatedly, which is inefficient for large exponents.

Python's built-in `pow(base, exponent, modulus)` uses efficient modular exponentiation.

In [13]:
def exp_mod_naive(x: int, e: int, n: int) -> int:
    """Very slow modular exponentiation for demonstration."""
    if e == 0:
        return 1 % n
    y = 1
    for _ in range(e):
        y = (y * x) % n
    return y


def exp_mod_square_and_multiply(x: int, e: int, n: int) -> int:
    """Square-and-multiply modular exponentiation."""
    result = 1
    base = x % n
    while e > 0:
        if e & 1:
            result = (result * base) % n
        base = (base * base) % n
        e >>= 1
    return result

x, exponent, modulus = 3, 65537, 36567232109354321
print(pow(x, exponent, modulus))
print(exp_mod_square_and_multiply(x, exponent, modulus))

26652909283612267
26652909283612267


### Exercise 6 — Compare exponentiation strategies

Time the naive function, the square-and-multiply function, and Python's built-in `pow()` for a moderate exponent such as `10_000` or `50_000`.

Do not choose an exponent so large that your notebook becomes slow.

In [20]:
# TODO: Compare runtimes.
base = 7
exponent = 10_000
modulus = 999_983

# Your timing code here
start = time.perf_counter()
print(exp_mod_naive(base, exponent, modulus))
elapsed = time.perf_counter() - start
print(f"  time: {elapsed:.6f} seconds")

start = time.perf_counter()
print(exp_mod_square_and_multiply(base, exponent, modulus))
elapsed = time.perf_counter() - start
print(f"  time: {elapsed:.6f} seconds")

start = time.perf_counter()
print(pow(base, exponent, modulus))
elapsed = time.perf_counter() - start
print(f"  time: {elapsed:.6f} seconds")

861514
  time: 0.001124 seconds
861514
  time: 0.000105 seconds
861514
  time: 0.000071 seconds


## Part 7 — RSA-OAEP using a library

Modern RSA encryption should use a padding scheme such as OAEP.

In real systems, RSA is often used to encrypt a symmetric key, while the actual data is encrypted with a symmetric authenticated cipher such as AES-GCM or ChaCha20-Poly1305.

In [21]:
try:
    from cryptography.hazmat.primitives.asymmetric import rsa, padding
    from cryptography.hazmat.primitives import hashes, serialization
    CRYPTOGRAPHY_AVAILABLE = True
except Exception as exc:
    CRYPTOGRAPHY_AVAILABLE = False
    print("cryptography package is not available:", exc)

CRYPTOGRAPHY_AVAILABLE

True

In [27]:
if CRYPTOGRAPHY_AVAILABLE:
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )
    public_key = private_key.public_key()

    symmetric_key = b"example AES key!!"  # 16 bytes; demonstration only

    ciphertext = public_key.encrypt(
        symmetric_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )

    recovered_key = private_key.decrypt(
        ciphertext,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )

    print("ciphertext length:", len(ciphertext))
    print("recovered key:", recovered_key)
    print("matches?", recovered_key == symmetric_key)

ciphertext length: 256
recovered key: b'example AES key!!'
matches? True


### Exercise 7 — OAEP behavior

Run the previous encryption cell more than once. Does the ciphertext change when encrypting the same `symmetric_key` under the same public key?

Explain why that behavior is desirable.

Answer:
- The padding makes it nondeterministic so it changes eveytime. This is good becasue it creates non predictability even if it is the same public key, it makes it harder for the attackers. 

## Part 8 — RSA-PSS signatures using a library

RSA signatures should use a signature scheme such as PSS. A signature is not the same as encryption.

Encryption aims to protect confidentiality. A signature aims to provide authenticity and help prevent forgery.

In [28]:
if CRYPTOGRAPHY_AVAILABLE:
    message = b"Approve transfer of $100.00"

    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH,
        ),
        hashes.SHA256(),
    )

    print("signature length:", len(signature))

    # Verification succeeds silently if the signature is valid.
    try:
        public_key.verify(
            signature,
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH,
            ),
            hashes.SHA256(),
        )
        print("signature verified")
    except:
        print("signature not valid")

signature length: 256
signature verified


### Exercise 8 — Signature failure

Modify the signed message by one byte and verify that the original signature no longer verifies.

Catch the exception and print a clear message.

In [32]:
# TODO: Demonstrate that changing the message invalidates the signature.
if CRYPTOGRAPHY_AVAILABLE:
    tampered_message = b"Approve transfer of $900.00"
    # Your verification code here
    try:
        public_key.verify(
            signature,
            tampered_message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH,
            ),
            hashes.SHA256(),
        )
        print("signature verified")
    except:
        print("signature not valid")

signature not valid


## Part 9 — Engineering checkpoint

Answer the following in your own words.

1. Why should textbook RSA not be used for encryption?
- Becasue it is deterministic and malleable making it extremely vunerable to commaon attacks. 
2. Why is OAEP used with RSA encryption?
- it adds randomness and padding to the raw RSa to make it unredictapble and less vunerable to attacks. 
3. Why is PSS used with RSA signatures?
- It is to add salt value to add security and improve upon the deterministic padding of traditional RSA. 
4. Why is e = 65537 common?
- In binary it only has two 1s and 0's in between. It is a good number for fast computation but also security. 
5. What should a security review check besides the word RSA?
- There should be checks for things such as the method of RSA being used such as textbook or extra RSA with PSS and OPEA. Also the key length and sizes as well as the prime numbers being used for security and computation. 

## Submission checklist

Before submitting, make sure you have:

- [ ] Completed all written responses.
- [ ] Completed all TODO code cells.
- [ ] Run the notebook from top to bottom without errors.
- [ ] Updated your engineering log.

## Final reflection

Return to today's big question:

> Why is “uses RSA” not enough information to know whether a system is secure?

Answer using at least three details from this notebook.

Relfection:
Uses RSA is not enough information to know wheter a system is secure becasue there are a lot of weakness for RSA. One of the problems is that it is not random enough for it to be secure. There also needs to have large enough prime numbers so that it can't be guess easily. It also depends on the length of the keys becasue if the key leghtn is not sufficient enough then it can be guessed easily. Textbook RSA is deterministic so the if you know the shared key than you can also guess at the private key. There needs to be the proper implementation of RSA with the up to date methods and libraries to properly secure information. 